# Tugas Mingdep — Ekstraksi Fitur Citra & Klasifikasi

**Kelompok:** (isi nama anggota)

**Kasus:** klasifikasi citra `berantakan` vs `tidak_berantakan` (±100 citra per kelas).

**Alur:**
1. Fitur warna per channel RGB: mean, std, skewness → `fitur_warna.csv`
2. Ubah citra ke grayscale
3. Fitur tekstur dari GLCM: energy, entropy, contrast, correlation, homogeneity
4. Gabung → `fitur_lengkap.csv` (14 fitur + label)
5. Bandingkan 5 model baseline (Stratified 5-fold CV), pakai yang terbaik
6. Evaluasi: accuracy + confusion matrix

**Acuan rumus:** Hallur & Gavade (2025), *Image feature extraction techniques: A comprehensive review*, Franklin Open 12, 100366.

## 1. Import & Konfigurasi
Ubah `DATASET_DIR` sesuai lokasi dataset. Struktur folder:

```
dataset/
├── berantakan/
│   ├── img001.jpg
│   └── ...
└── tidak_berantakan/
    ├── img001.jpg
    └── ...
```
Nama folder dipakai sebagai label.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

from skimage.feature import graycomatrix, graycoprops

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# ---- Konfigurasi ----
RANDOM_STATE = 42
DATASET_DIR = "dataset"          # ganti sesuai lokasi dataset
IMG_SIZE = (256, 256)            # resize agar fitur tidak dipengaruhi resolusi; None = tanpa resize
CSV_WARNA = "fitur_warna.csv"
CSV_FINAL = "fitur_lengkap.csv"

# ---- Parameter GLCM (jurnal tidak menetapkan, ditentukan kelompok) ----
GLCM_LEVELS = 256
GLCM_DISTANCES = [1]
GLCM_ANGLES = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]   # 0, 45, 90, 135 derajat (hasil dirata-rata)

TARGET_AKURASI = 0.90            # garis target pada grafik (opsional)

# ---- Jika memakai Google Colab, aktifkan 3 baris ini ----
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_DIR = "/content/drive/MyDrive/<folder_dataset_kalian>"

## 2. Muat Daftar Citra

In [ ]:
EXT = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")

def list_images(root):
    data = []
    for label in sorted(os.listdir(root)):
        folder = os.path.join(root, label)
        if not os.path.isdir(folder):
            continue
        for fname in sorted(os.listdir(folder)):
            if fname.lower().endswith(EXT):
                data.append((os.path.join(folder, fname), label))
    return data

files = list_images(DATASET_DIR)
assert len(files) > 0, f"Tidak ada citra ditemukan di '{DATASET_DIR}'. Cek path dan struktur folder."

df_files = pd.DataFrame(files, columns=["path", "label"])
print("Total citra:", len(df_files))
print(df_files["label"].value_counts())

## 3. Fitur Warna per Channel (RGB)

Dihitung terpisah untuk channel R, G, B. Rumus dari jurnal:

| Fitur | Eq. | Rumus |
|---|---|---|
| Mean | (2) | $\mu_c = \frac{1}{M \times N}\sum_{i=1}^{M}\sum_{j=1}^{N} C(i,j)$ |
| Std | (3) | $\sigma = \sqrt{\frac{1}{N}\sum (x_i-\mu)^2}$ (pembagi N, std populasi) |
| Skewness | (4) | $\frac{1}{M \times N}\sum_{i}\sum_{j}\left(\frac{I(i,j)-\mu}{\sigma}\right)^3$ |

In [ ]:
def load_rgb(path):
    img = Image.open(path).convert("RGB")
    if IMG_SIZE:
        img = img.resize(IMG_SIZE)
    return np.asarray(img, dtype=np.float64)

def channel_stats(ch):
    mu = ch.mean()                                   # Eq. (2)
    sigma = np.sqrt(np.mean((ch - mu) ** 2))         # Eq. (3)  pembagi N
    skew = 0.0 if sigma == 0 else np.mean(((ch - mu) / sigma) ** 3)   # Eq. (4)
    return mu, sigma, skew

rows = []
for path, label in tqdm(files, desc="Fitur warna"):
    rgb = load_rgb(path)
    stats = [channel_stats(rgb[:, :, k]) for k in range(3)]   # R, G, B
    row = {"filename": os.path.basename(path)}
    for k, ch in enumerate("RGB"):
        row[f"mean_{ch}"] = stats[k][0]
    for k, ch in enumerate("RGB"):
        row[f"std_{ch}"] = stats[k][1]
    for k, ch in enumerate("RGB"):
        row[f"skew_{ch}"] = stats[k][2]
    row["label"] = label
    rows.append(row)

df_warna = pd.DataFrame(rows)
df_warna.to_csv(CSV_WARNA, index=False)
print(f"Tersimpan: {CSV_WARNA}  |  bentuk: {df_warna.shape}")
df_warna.head()

## 4. Konversi ke Grayscale
Konversi memakai luminance standar: $0.299R + 0.587G + 0.114B$ (`PIL` mode `L`).

In [ ]:
def load_gray(path):
    img = Image.open(path).convert("L")
    if IMG_SIZE:
        img = img.resize(IMG_SIZE)
    return np.asarray(img, dtype=np.uint8)

# Contoh sebelum / sesudah: 2 citra pertama tiap kelas
sample = df_files.groupby("label").head(2).reset_index(drop=True)
fig, axes = plt.subplots(2, len(sample), figsize=(3 * len(sample), 6))
for i, r in sample.iterrows():
    axes[0, i].imshow(load_rgb(r["path"]).astype(np.uint8))
    axes[0, i].set_title(r["label"], fontsize=9)
    axes[1, i].imshow(load_gray(r["path"]), cmap="gray", vmin=0, vmax=255)
    for a in (axes[0, i], axes[1, i]):
        a.axis("off")
axes[0, 0].text(-0.05, 0.5, "RGB", transform=axes[0, 0].transAxes, rotation=90, va="center", ha="right")
axes[1, 0].text(-0.05, 0.5, "Grayscale", transform=axes[1, 0].transAxes, rotation=90, va="center", ha="right")
plt.tight_layout(); plt.show()

## 5. Fitur Tekstur GLCM

GLCM dihitung dari citra grayscale, lalu **dinormalisasi** (tiap elemen dibagi total elemen) menjadi $P(i,j)$.

| Fitur | Eq. | Rumus |
|---|---|---|
| Energy | (8) | $\sum\sum P(i,j)^2$ |
| Entropy | (5) | $-\sum\sum P(i,j)\log_2 P(i,j)$ |
| Contrast | (6) | $\sum\sum (i-j)^2 \, P(i,j)$ |
| Correlation | (7) | $\sum\sum \frac{(i-\mu_i)(j-\mu_j)\,P(i,j)}{\sigma_i\sigma_j}$ |
| Homogeneity | (9) | $\sum\sum \frac{P(i,j)}{1+\lvert i-j\rvert}$ |

> Dihitung **manual** dari matriks $P$ karena `graycoprops` berbeda dari jurnal:
> `energy` di scikit-image = $\sqrt{ASM}$ (jurnal: $\sum P^2$), `homogeneity` memakai $(i-j)^2$ (jurnal: $\lvert i-j\rvert$), dan entropy tidak tersedia.

In [ ]:
def glcm_features(gray, levels=GLCM_LEVELS, distances=GLCM_DISTANCES, angles=GLCM_ANGLES):
    """Hitung 5 fitur GLCM sesuai Eq. (5)-(9) jurnal; hasil dirata-rata pada semua jarak & sudut."""
    g = gray
    if levels < 256:   # kuantisasi opsional jika ingin level lebih kecil
        g = (gray.astype(np.float64) / 256 * levels).astype(np.uint8)

    glcm = graycomatrix(g, distances=distances, angles=angles,
                        levels=levels, symmetric=True, normed=False)

    idx = np.arange(levels, dtype=np.float64)
    I = idx.reshape(-1, 1)
    J = idx.reshape(1, -1)
    W_contrast = (I - J) ** 2
    W_homog = 1.0 / (1.0 + np.abs(I - J))

    out = {"energy": [], "entropy": [], "contrast": [], "correlation": [], "homogeneity": []}
    for d in range(glcm.shape[2]):
        for a in range(glcm.shape[3]):
            P = glcm[:, :, d, a].astype(np.float64)
            P = P / P.sum()                                   # normalisasi -> probabilitas
            nz = P > 0

            mu_i, mu_j = np.sum(I * P), np.sum(J * P)
            sd_i = np.sqrt(np.sum((I - mu_i) ** 2 * P))
            sd_j = np.sqrt(np.sum((J - mu_j) ** 2 * P))

            out["energy"].append(np.sum(P ** 2))                          # Eq. (8)
            out["entropy"].append(-np.sum(P[nz] * np.log2(P[nz])))        # Eq. (5)
            out["contrast"].append(np.sum(W_contrast * P))                # Eq. (6)
            if sd_i * sd_j == 0:                                          # citra seragam: korelasi tak terdefinisi
                out["correlation"].append(1.0)
            else:
                out["correlation"].append(np.sum((I - mu_i) * (J - mu_j) * P) / (sd_i * sd_j))   # Eq. (7)
            out["homogeneity"].append(np.sum(W_homog * P))                # Eq. (9)

    return {k: float(np.mean(v)) for k, v in out.items()}

### Cek kebenaran (sanity check)
Bandingkan hasil manual dengan `graycoprops` pada 1 citra, 1 jarak, 1 sudut. **Contrast, energy (ASM), dan correlation harus sama.** Homogeneity memang berbeda (rumus jurnal ≠ rumus scikit-image).

In [ ]:
g0 = load_gray(files[0][0])
man = glcm_features(g0, distances=[1], angles=[0])

glcm_n = graycomatrix(g0, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
cek = pd.DataFrame({
    "manual (jurnal)": [man["contrast"], man["energy"], man["correlation"], man["homogeneity"]],
    "graycoprops":     [graycoprops(glcm_n, "contrast")[0, 0], graycoprops(glcm_n, "ASM")[0, 0],
                        graycoprops(glcm_n, "correlation")[0, 0], graycoprops(glcm_n, "homogeneity")[0, 0]],
}, index=["contrast", "energy (= ASM)", "correlation", "homogeneity (beda rumus)"])
cek

In [ ]:
rows_tex = []
for path, label in tqdm(files, desc="Fitur GLCM"):
    rows_tex.append(glcm_features(load_gray(path)))

df_tex = pd.DataFrame(rows_tex)
df_tex.head()

## 6. Gabung Fitur → CSV Final (14 fitur + label)

In [ ]:
FITUR = ["mean_R", "mean_G", "mean_B",
         "std_R", "std_G", "std_B",
         "skew_R", "skew_G", "skew_B",
         "energy", "entropy", "contrast", "correlation", "homogeneity"]

df_final = pd.concat([df_warna.drop(columns="label").reset_index(drop=True),
                      df_tex.reset_index(drop=True)], axis=1)
df_final["label"] = df_warna["label"].values
df_final = df_final[["filename"] + FITUR + ["label"]]

df_final.to_csv(CSV_FINAL, index=False)
print(f"Tersimpan: {CSV_FINAL}  |  bentuk: {df_final.shape}  |  jumlah fitur: {len(FITUR)}")
df_final.head()

## 7. Eksplorasi Singkat

In [ ]:
print("Missing value per kolom:\n", df_final.isna().sum()[df_final.isna().sum() > 0])
print("Tidak ada missing value." if df_final.isna().sum().sum() == 0 else "")
print("\nJumlah per kelas:\n", df_final["label"].value_counts())
df_final[FITUR].describe().T

In [ ]:
# Boxplot fitur tekstur per kelas
tex_cols = ["energy", "entropy", "contrast", "correlation", "homogeneity"]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))
for ax, c in zip(axes, tex_cols):
    sns.boxplot(data=df_final, x="label", y=c, ax=ax, palette="Set2")
    ax.set_title(c); ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

## 8. Perbandingan 5 Model Baseline
Semua model dievaluasi dengan **Stratified 5-fold cross-validation** pada seluruh data (metrik: accuracy).
`StandardScaler` dimasukkan ke dalam pipeline untuk SVM, Logistic Regression, dan K-NN agar tidak terjadi kebocoran data.

> Samakan 5 model ini dengan yang dipakai di tugas audio. Jika berbeda, ubah dictionary `baseline_models`.

In [ ]:
X_all = df_final[FITUR].values
le = LabelEncoder()
y_all = le.fit_transform(df_final["label"])
print("Kelas:", dict(zip(le.classes_, le.transform(le.classes_))))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

baseline_models = {
    "SVM (RBF)":           make_pipeline(StandardScaler(), SVC(C=10, gamma=0.01)),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, C=0.5)),
    "Random Forest":       RandomForestClassifier(500, random_state=RANDOM_STATE, n_jobs=-1),
    "Extra Trees":         ExtraTreesClassifier(500, random_state=RANDOM_STATE, n_jobs=-1),
    "K-NN (k=5)":          make_pipeline(StandardScaler(), KNeighborsClassifier(5, weights="distance")),
}

hasil = {}
for nama, model in baseline_models.items():
    skor = cross_val_score(model, X_all, y_all, cv=cv, scoring="accuracy")
    hasil[nama] = skor
    print(f"{nama:22s} akurasi CV = {skor.mean():.4f} +/- {skor.std():.4f}")

In [ ]:
df_hasil = pd.DataFrame(hasil)
ringkasan = pd.DataFrame({"akurasi_mean": df_hasil.mean(), "akurasi_std": df_hasil.std()}).sort_values("akurasi_mean", ascending=False)
display(ringkasan.round(4))

fig, ax = plt.subplots(figsize=(11, 4.5))
df_hasil.mean().sort_values().plot.barh(ax=ax, color="#16a085")
ax.axvline(TARGET_AKURASI, color="red", ls="--", lw=1, label=f"target {int(TARGET_AKURASI*100)}%")
ax.set_xlim(0.5, 1.0); ax.set_xlabel("akurasi CV"); ax.legend()
ax.set_title("Perbandingan model baseline")
plt.tight_layout(); plt.show()

best_name = df_hasil.mean().idxmax()
print("Model terbaik:", best_name)

## 9. Matriks Akurasi (Confusion Matrix) — Semua Model
Confusion matrix dari prediksi cross-validation (tiap citra diprediksi saat menjadi data uji pada fold-nya).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()
for ax, (nama, model) in zip(axes, baseline_models.items()):
    y_cv = cross_val_predict(model, X_all, y_all, cv=cv)
    acc = accuracy_score(y_all, y_cv)
    ConfusionMatrixDisplay.from_predictions(y_all, y_cv, display_labels=le.classes_,
                                            cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(f"{nama}\nacc = {acc:.4f}")
for ax in axes[len(baseline_models):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 10. Evaluasi Model Terbaik (Holdout 80/20)
Data dibagi 80/20 (stratified). Model terbaik dilatih pada data train, lalu diuji pada data test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=RANDOM_STATE)
print(f"Train: {len(y_train)} | Test: {len(y_test)}")

model_terbaik = clone(baseline_models[best_name])
model_terbaik.fit(X_train, y_train)
y_pred = model_terbaik.predict(X_test)

akurasi_test = accuracy_score(y_test, y_pred)
print(f"Model: {best_name}")
print(f"Akurasi holdout test = {akurasi_test:.4f} ({akurasi_test * 100:.2f}%)\n")
print(classification_report(y_test, y_pred, target_names=le.classes_, digits=3))

fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=le.classes_, cmap="Blues", ax=ax)
ax.set_title(f"Confusion Matrix — {best_name}\n(data test)")
plt.tight_layout(); plt.show()

## 11. Kesimpulan & Analisis
*(isi oleh kelompok setelah hasil keluar)*

- **Model terbaik:** ...  (accuracy CV = ..., accuracy holdout = ...)
- **Perbandingan model:** model mana yang unggul/tertinggal dan kemungkinan alasannya.
- **Kesalahan klasifikasi:** dari confusion matrix, kelas mana yang sering tertukar?
- **Fitur:** fitur mana yang tampak membedakan `berantakan` vs `tidak_berantakan` (lihat boxplot bagian 7)?
- **Keterbatasan:** jumlah data ±200 (hasil CV bervariasi), parameter GLCM yang dipilih, kriteria pelabelan subjektif.

### Pembagian tugas
| Bagian | Penanggung jawab |
|---|---|
| Dataset & pelabelan | |
| Fitur warna | |
| Fitur GLCM | |
| Klasifikasi & evaluasi | |
| Laporan | |